# Imports

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), os.pardir))

In [3]:
from Tools.leica_tools import RawLoader, parse_lif
from Tools.sample_tools import Sample
from Tools.db_tools import DbManager


EXP_DIR: /Users/xiangxigao/github/droplet-phenotyping/Experiments


2026-07-09 18:12:53.943086: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
parse_lif('/Volumes/Externe_Festplatte_1/NKQC/NKQC_JB_005.lif')

,index,name,timestamp,t_index,n_channels,bit_depth,resolution,merged
0,0,TileScan 1/Mix Merged,2025-05-08 15:52:42,0,4,16,1.527159,True
1,1,TileScan 1/NK_single Merged,2025-05-08 15:59:36,0,4,16,1.527159,True
2,2,TileScan 2/Mix Merged,2025-05-08 17:12:30,0,4,16,1.527159,True
3,2,TileScan 2/Mix Merged,2025-05-08 18:12:30,1,4,16,1.527159,True
4,2,TileScan 2/Mix Merged,2025-05-08 19:12:30,2,4,16,1.527159,True
5,2,TileScan 2/Mix Merged,2025-05-08 20:12:30,3,4,16,1.527159,True
6,2,TileScan 2/Mix Merged,2025-05-08 21:12:30,4,4,16,1.527159,True
7,2,TileScan 2/Mix Merged,2025-05-08 22:12:30,5,4,16,1.527159,True
8,3,TileScan 2/NK_single Merged,2025-05-08 17:19:15,0,4,16,1.527160,True
9,3,TileScan 2/NK_single Merged,2025-05-08 18:19:15,1,4,16,1.527160,True


# Data prep

In [13]:
expID = 'NKQC_JB_005'
rawloader = RawLoader(expID)
rawloader.frame_df

,droplet_size,size_range,image_index,t_index,time,condition,path
frameID,,,,,,,
0,90,5,0,0,1,Mix,/Volumes/Externe_Festplatte_1/NKQC/NKQC_JB_005...


# Droplet detection

Execute a preview run of the droplet detection. An image will be saved to the exp folder in the analyses directory. 
If droplets are not well detected consider changing the droplet size estimate in setup.xlsx (re-run RawLoader API).

In [15]:
frameID = 0
sample = Sample(expID, frameID)
sample.detect_droplets(mode='sweep')
sample.visualize_droplets(channel=0,save=True)

2134 droplets in frame 0 detected 



Run droplet detection through all frames of the experiment. drop_register.csv will be created at the end of the process.

In [14]:
dbm = DbManager()
dbm.detect_droplets(expID, mode='sweep')

2134 droplets in frame 0 detected 



# Outlier detection

In [16]:
# expID = 'NKIP_FA_084'
dbm = DbManager()
dbm.detect_outliers(expID, model_name='outlier_v3.h5')

67/67 ━━━━━━━━━━━━━━━━━━━━ 7s 97ms/step


2026-07-10 17:42:31.349815: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.11/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)
2026-07-10 17:42:32.127567: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-07-10 17:42:33.204636: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-07-10 17:42:33.565639: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-07-10 17:42:34.736343: W te

In [18]:
sample = Sample(expID, 0)
sample.reload_droplets()
sample.visualize_droplets(channel=0)

# Workpackage Generation

In [3]:
dbm = DbManager()
dbm.generate_wp(expID='NKIP_FA_065', exclude_query='outlier == True')

2024-09-08 22:10:21.239744: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.339381: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.549281: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.961074: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:22.772556: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Cell counting

In [19]:
dbm = DbManager()
dbm.cell_count(expID=expID, model_name='cell_count_v3.h5')

2026-07-10 17:45:31.678355: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


67/67 ━━━━━━━━━━━━━━━━━━━━ 20s 287ms/step


/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.11/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)
2026-07-10 17:45:51.251220: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
